# SAM2 (zero-shot then fine-tuned) - BTXRD

Single file, two parts, run top to bottom:
- **Part A**: official pretrained `sam2.1_hiera_large` checkpoint, box prompt,
  no fine-tuning - zero-shot reference number.
- **Part B**: fine-tunes the prompt encoder and mask decoder on this dataset's
  training split (image encoder frozen, following the standard SAM2
  fine-tuning recipe), model-selects on the validation split under the
  off-center condition (matching the article's own convention for every other
  model), then re-tests the best checkpoint the same way as Part A.

Both parts use the exact same box protocol as every other model in this
article (`_center_zoom_bbox` = covering, `_center_shift_bbox` = off-center),
reused directly from `PromptSegmentationDataset` in the PGA-UNet codebase, and
report the same six metrics (Dice, IoU, Precision, Recall, HD95, CBL).

In [ ]:
# -- Setup -----------------------------------------------------------------
%cd /kaggle/working
import os, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# PGA-UNet repo: only needed for its dataset.py (box protocol + polygon listing).
if not os.path.exists('PGA_Unet2D'):
    !git clone --branch main --single-branch https://github.com/ThongLuc2k3/PGA_Unet2D.git
PGA_PATH = '/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# SAM2 repo (official facebookresearch implementation)
if not os.path.exists('sam2'):
    !git clone --branch main --single-branch https://github.com/facebookresearch/sam2.git
%cd /kaggle/working/sam2
!pip install -q -e .

SAM2_CKPT_NAME = 'sam2.1_hiera_large.pt'
SAM2_CFG       = 'configs/sam2.1/sam2.1_hiera_l.yaml'
if not os.path.exists(f'checkpoints/{SAM2_CKPT_NAME}'):
    !wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/{SAM2_CKPT_NAME}
assert os.path.exists(f'checkpoints/{SAM2_CKPT_NAME}')
SAM2_CKPT_PATH = os.path.abspath(f'checkpoints/{SAM2_CKPT_NAME}')

import gdown
DATASET_ID   = '1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx'
DATASET_ROOT = 'dataset_BTXRD'
if not os.path.exists(f'/kaggle/working/{DATASET_ROOT}'):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}',
                   f'/kaggle/working/{DATASET_ROOT}.zip', quiet=False)
    !unzip -oq /kaggle/working/{DATASET_ROOT}.zip -d /kaggle/working/
print(f'\nSetup complete | dataset=BTXRD | checkpoint {os.path.getsize(SAM2_CKPT_PATH)//1024//1024} MB')

In [ ]:
# -- Model + shared helpers -------------------------------------------------
import sys, csv, json as _json, random
import numpy as np
import cv2
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt

if PGA_PATH not in sys.path:
    sys.path.insert(0, PGA_PATH)
from dataset import PromptSegmentationDataset

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DS_NAME = 'BTXRD'
SEED = 22120196
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

TRAIN_IMG  = f'{PGA_PATH}/{DATASET_ROOT}/train/images'
TRAIN_JSON = f'{PGA_PATH}/{DATASET_ROOT}/train/annotations'
VAL_IMG    = f'{PGA_PATH}/{DATASET_ROOT}/val/images'
VAL_JSON   = f'{PGA_PATH}/{DATASET_ROOT}/val/annotations'
TEST_IMG   = f'{PGA_PATH}/{DATASET_ROOT}/test/images'
TEST_JSON  = f'{PGA_PATH}/{DATASET_ROOT}/test/annotations'
RESULT_DIR = f'{PGA_PATH}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

sam2_model = build_sam2(SAM2_CFG, SAM2_CKPT_PATH, device=DEVICE)
predictor = SAM2ImagePredictor(sam2_model)

# One box-protocol dataset per split: is_train=True gives PromptSegmentationDataset's
# own random (not seeded) center_shift jitter for training; is_train=False gives the
# fixed, reproducible per-sample offset used for validation and test everywhere else.
test_box_ds = PromptSegmentationDataset(TEST_IMG, TEST_JSON, is_train=False, prompt_mode='center_zoom')
print(f'{DS_NAME}: {len(test_box_ds.all_samples)} (image, polygon) test samples')

@torch.no_grad()
def sam2_predict_mask(box_xyxy):
    # predictor.set_image(...) must already have been called for the current image.
    # Returns a boolean mask at the image's ORIGINAL resolution. Uses predictor.model's
    # LIVE weights, so this automatically reflects fine-tuning done in Part B.
    masks, scores, _ = predictor.predict(
        box=np.array(box_xyxy, dtype=np.float32)[None, :],
        multimask_output=False,
    )
    return masks[0].astype(bool)

def run_test(box_ds, img_dir, json_dir, tag):
    # Shared by Part A (zero-shot) and Part B (post-fine-tune): runs both prompt
    # conditions over every test image, merges polygons by logical OR (equivalent
    # to pixelwise-max-then-threshold since predict() already returns binary masks),
    # and reports/saves the six standard metrics.
    predictor.model.sam_mask_decoder.eval()
    predictor.model.sam_prompt_encoder.eval()
    by_image = {}
    names = sorted(set(n for n, _ in box_ds.all_samples))
    for n_done, img_name in enumerate(names):
        base = os.path.splitext(img_name)[0]
        img_gray = cv2.imread(os.path.join(img_dir, img_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        img_3c = np.repeat(img_gray[:, :, None], 3, axis=-1)
        predictor.set_image(img_3c)
        with open(os.path.join(json_dir, base + '.json'), encoding='utf-8') as f:
            data = _json.load(f)
        pred_zoom  = np.zeros((H, W), dtype=bool)
        pred_shift = np.zeros((H, W), dtype=bool)
        gt_union   = np.zeros((H, W), dtype=np.uint8)
        poly_idxs = [i for i, s in enumerate(data.get('shapes', [])) if s.get('shape_type') == 'polygon']
        for shape_idx in poly_idxs:
            points = np.array(data['shapes'][shape_idx]['points'])
            cv2.fillPoly(gt_union, [points.astype(np.int32)], 1)
            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            sample_idx = box_ds.all_samples.index((img_name, shape_idx))
            bz = box_ds._center_zoom_bbox(x_min, x_max, y_min, y_max, H, W)
            bs = box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W, seed_idx=sample_idx)
            box_zoom  = [bz[0], bz[2], bz[1], bz[3]]
            box_shift = [bs[0], bs[2], bs[1], bs[3]]
            pred_zoom  |= sam2_predict_mask(box_zoom)
            pred_shift |= sam2_predict_mask(box_shift)
        by_image[img_name] = dict(pred_zoom=pred_zoom, pred_shift=pred_shift, gt=gt_union)
        if (n_done + 1) % 25 == 0:
            print(f'  {n_done + 1}/{len(names)} images')
    rows = []
    for mode, key in [('covering', 'pred_zoom'), ('off-center', 'pred_shift')]:
        per_image = []
        for img_name, rec in by_image.items():
            m = calc_metrics_img(rec[key].astype(np.uint8), rec['gt'])
            m['image'] = img_name
            per_image.append(m)
        agg = {k: float(np.mean([r[k] for r in per_image])) for k in ('dice', 'iou', 'precision', 'recall', 'hd95', 'cbl')}
        rows.append(dict(dataset=DS_NAME, model=f'SAM2 ({tag})', prompt=mode, **agg))
        with open(f'{RESULT_DIR}/sam2_{tag.replace(chr(32),chr(95))}_{DS_NAME.lower()}_{mode.replace(chr(45),chr(95))}_per_image.csv',
                  'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(per_image[0].keys())); w.writeheader(); w.writerows(per_image)
    print(f'\n{DS_NAME}: SAM2 {tag}, image-level merged\n')
    print(f'{"prompt":<12}{"Dice":>8}{"IoU":>8}{"Prec":>8}{"Recall":>8}{"HD95":>8}{"CBL":>8}')
    print('-' * 60)
    for r in rows:
        print(f'{r["prompt"]:<12}{r["dice"]:>8.3f}{r["iou"]:>8.3f}{r["precision"]:>8.3f}'
              f'{r["recall"]:>8.3f}{r["hd95"]:>8.1f}{r["cbl"]:>8.3f}')
    with open(f'{RESULT_DIR}/sam2_{tag.replace(chr(32),chr(95))}_{DS_NAME.lower()}_summary.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return rows, by_image

# -- Metrics: Dice, IoU, Precision, Recall, HD95, CBL (six metrics) --
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any():
        return 0.0
    S = max(pred.shape)
    if not p.any() or not g.any():
        return float(S)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(S) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img(pred_bin, gt_bin, eps=1e-6):
    pm, gm = pred_bin.astype(np.float32), gt_bin.astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float(tp/(tp+fp+eps)), recall=float(tp/(tp+fn+eps)), hd95=hd95, cbl=cbl)

print('helpers ready | device', DEVICE)
# -- Qualitative visualization: same 5-column style as
# Finetune_SAMMed2D_test_robust.ipynb (Input / Prompt / Prediction / GT / TP-FP-FN),
# 10 shared stems, one PNG + one export_qualitative_rows call per stem.
def visualize_qualitative(by_image_vis, img_dir, box_ds_vis, model_label, prefix_tag):
    import sys as _sys
    from pathlib import Path as _Path
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    if str(_Path(PGA_PATH)) not in _sys.path:
        _sys.path.insert(0, str(_Path(PGA_PATH)))
    from qualitative_visualization import export_qualitative_rows, select_shared_stems

    modes = [('covering', 'pred_zoom', 'limegreen'), ('off-center', 'pred_shift', 'tomato')]
    selection_records = [dict(img_name=n) for n in by_image_vis.keys()]
    vis_images = select_shared_stems(selection_records, n_multi=5, n_single=5)

    for vis_name in vis_images:
        if vis_name not in by_image_vis:
            continue
        rec = by_image_vis[vis_name]
        img_gray = cv2.imread(os.path.join(img_dir, vis_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
        gt = rec['gt']
        ys, xs = np.where(gt > 0)
        gt_ov = rgb.copy()
        gt_ov[gt > 0] = np.clip(rgb[gt > 0] * 0.4 + np.array([0, 200, 0]) * 0.6, 0, 255)

        fig, axes = plt.subplots(len(modes), 5, figsize=(20, 4 * len(modes)))
        fig.suptitle(f'{model_label}: {vis_name}', fontsize=13, fontweight='bold', y=1.01)
        mode_records = []
        for row, (mode, key, color) in enumerate(modes):
            pred = rec[key].astype(np.uint8)
            pr_ov = rgb.copy()
            pr_ov[pred > 0] = np.clip(rgb[pred > 0] * 0.4 + np.array([220, 60, 60]) * 0.6, 0, 255)
            diff = rgb.copy()
            diff[gt > 0] = [0, 200, 0]
            diff[pred > 0] = [200, 60, 60]
            diff[(gt > 0) & (pred > 0)] = [220, 200, 0]

            axes[row, 0].imshow(img_gray, cmap='gray')
            axes[row, 0].set_ylabel(mode, fontsize=11, fontweight='bold', color=color,
                                    rotation=0, labelpad=65, va='center')
            axes[row, 1].imshow(img_gray, cmap='gray')
            if len(xs) > 0:
                x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
                if mode == 'covering':
                    bx0, bx1, by0, by1 = box_ds_vis._center_zoom_bbox(x0, x1, y0, y1, H, W)
                else:
                    bx0, bx1, by0, by1 = box_ds_vis._center_shift_bbox(x0, x1, y0, y1, H, W, seed_idx=row)
                axes[row, 1].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                                  linewidth=2.5, edgecolor=color, facecolor='none'))
            axes[row, 2].imshow(pr_ov)
            axes[row, 3].imshow(gt_ov)
            axes[row, 4].imshow(diff)
            mode_records.append(dict(img_name=f'{vis_name}_{mode}', gt=gt.copy(), pred=pred.copy()))
            for ax in axes[row]:
                ax.axis('off')

        plt.tight_layout()
        out = f'{prefix_tag}_{os.path.splitext(vis_name)[0]}.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        export_qualitative_rows(fig, axes, mode_records, prefix=f'{prefix_tag}_{vis_name}')
        print(f'saved {out}')


## Part A - zero-shot test (no fine-tuning)

Official pretrained weights as downloaded, box prompt only, both prompt
conditions, image-level merged.

In [ ]:
# -- Part A: run + report --
rows_zeroshot, images_zeroshot = run_test(test_box_ds, TEST_IMG, TEST_JSON, tag='zero-shot')

# -- Qualitative figures (10 shared stems, same style as Finetune_SAMMed2D_test_robust.ipynb) --
visualize_qualitative(images_zeroshot, TEST_IMG, test_box_ds, 'SAM2 (zero-shot)',
                      f'sam2_zeroshot_{DS_NAME.lower()}')

## Part B - fine-tune (prompt encoder + mask decoder, image encoder frozen)

Standard SAM2 box-prompt fine-tuning recipe: freeze the Hiera image
encoder, train the prompt encoder and mask decoder. Boxes follow the same
protocol as PGA-UNet and the fine-tuned SAM-Med2D baseline (`center_mixed`,
80% `center_shift` / 20% `center_zoom` per sample); SAM2 otherwise keeps
its own official recipe, and no flip or rotation augmentation is added
here. One optimizer step per 4-polygon batch (one polygon per predictor
call, since the image predictor holds a single active image). Model
selection is on the validation split under the fixed off-center
(`center_shift`) condition, early stopping with patience 30 (the
fine-tuned SAM-Med2D baseline's longer window, not PGA-UNet's from-scratch
15), up to 150 epochs. Learning rate and weight decay follow the common
SAM2 fine-tuning recipe (`lr=1e-5`, `weight_decay=4e-5`), distinct from
PGA-UNet's from-scratch hyperparameters since this fine-tunes a large
pretrained model.

In [ ]:
# -- Part B: fine-tuning setup --
import random

EPOCHS     = 150
PATIENCE   = 30     # matches the fine-tuned SAM-Med2D baseline in this article, not PGA's own from-scratch 15
BATCH_SIZE = 4      # matches PGA-UNet's and SAM-Med2D's batch size (loss averaged over 4 samples per step)
LR         = 1e-5
WD         = 4e-5
MIXED_SHIFT_PROB = 0.8   # matches PromptSegmentationDataset's own default
BEST_CKPT_PATH = f'{PGA_PATH}/checkpoints/sam2_finetuned_{DS_NAME.lower()}_best.pt'
os.makedirs(os.path.dirname(BEST_CKPT_PATH), exist_ok=True)

train_box_ds = PromptSegmentationDataset(TRAIN_IMG, TRAIN_JSON, is_train=True, prompt_mode='center_mixed')
val_box_ds   = PromptSegmentationDataset(VAL_IMG, VAL_JSON, is_train=False, prompt_mode='center_shift')
print(f'{DS_NAME}: {len(train_box_ds.all_samples)} train polygons, {len(val_box_ds.all_samples)} val polygons')

# Freeze SAM2's heavy Hiera image encoder; train only the prompt encoder +
# mask decoder (standard SAM2 box-prompt fine-tuning recipe, mirroring the
# SAM-Med2D baseline's explicit "image_encoder.requires_grad = False" setup).
for _name, _p in predictor.model.named_parameters():
    _p.requires_grad = ('sam_mask_decoder' in _name) or ('sam_prompt_encoder' in _name)
if hasattr(predictor.model, 'image_encoder'):
    predictor.model.image_encoder.eval()
predictor.model.sam_mask_decoder.train(True)
predictor.model.sam_prompt_encoder.train(True)
_trainable = [p for p in predictor.model.parameters() if p.requires_grad]
_total = sum(p.numel() for p in predictor.model.parameters())
print(f'trainable: {sum(p.numel() for p in _trainable) / 1e6:.1f}M / {_total / 1e6:.1f}M total')
optimizer = torch.optim.AdamW(params=_trainable, lr=LR, weight_decay=WD)
scaler = torch.cuda.amp.GradScaler()

def sam2_compute_loss(img_gray, box_xyxy, gt_mask_hw):
    # Returns the loss TENSOR (no backward/step here) so the training loop can
    # average it with 3 other samples first, matching PGA/SAM-Med2D's batch_size=4.
    # One polygon per call (predictor.set_image only holds one active image), unlike
    # the reference script's per-image multi-polygon batching.
    img_3c = np.repeat(img_gray[:, :, None], 3, axis=-1)
    gt_mask = torch.tensor(gt_mask_hw.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.cuda.amp.autocast():
        predictor.set_image(img_3c)
        box = np.array([box_xyxy], dtype=np.float32)
        _, _, _, unnorm_box = predictor._prep_prompts(
            point_coords=None, point_labels=None, box=box, mask_logits=None, normalize_coords=True)
        sparse_emb, dense_emb = predictor.model.sam_prompt_encoder(points=None, boxes=unnorm_box, masks=None)
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features['high_res_feats']]
        low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
            image_embeddings=predictor._features['image_embed'][-1].unsqueeze(0),
            image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
            repeat_image=False,
            high_res_features=high_res_features,
        )
        prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])
        prd_mask = torch.sigmoid(prd_masks[:, 0])
        seg_loss = (-gt_mask * torch.log(prd_mask + 1e-5) - (1 - gt_mask) * torch.log((1 - prd_mask) + 1e-5)).mean()
        inter = (gt_mask * (prd_mask > 0.5)).sum(dim=(1, 2))
        iou = inter / (gt_mask.sum(dim=(1, 2)) + (prd_mask > 0.5).sum(dim=(1, 2)) - inter + 1e-5)
        score_loss = torch.abs(prd_scores[:, 0] - iou).mean()
    return seg_loss + score_loss * 0.05

@torch.no_grad()
def validate():
    predictor.model.sam_mask_decoder.eval()
    predictor.model.sam_prompt_encoder.eval()
    names = sorted(set(n for n, _ in val_box_ds.all_samples))
    dices = []
    for img_name in names:
        base = os.path.splitext(img_name)[0]
        img_gray = cv2.imread(os.path.join(VAL_IMG, img_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        img_3c = np.repeat(img_gray[:, :, None], 3, axis=-1)
        predictor.set_image(img_3c)
        with open(os.path.join(VAL_JSON, base + '.json'), encoding='utf-8') as f:
            data = _json.load(f)
        pred = np.zeros((H, W), dtype=bool)
        gt = np.zeros((H, W), dtype=np.uint8)
        poly_idxs = [i for i, s in enumerate(data.get('shapes', [])) if s.get('shape_type') == 'polygon']
        for shape_idx in poly_idxs:
            points = np.array(data['shapes'][shape_idx]['points'])
            cv2.fillPoly(gt, [points.astype(np.int32)], 1)
            x_min, y_min = points.min(axis=0); x_max, y_max = points.max(axis=0)
            sample_idx = val_box_ds.all_samples.index((img_name, shape_idx))
            b = val_box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W, seed_idx=sample_idx)
            box_xyxy = [b[0], b[2], b[1], b[3]]
            pred |= sam2_predict_mask(box_xyxy)
        m = calc_metrics_img(pred.astype(np.uint8), gt)
        dices.append(m['dice'])
    predictor.model.sam_mask_decoder.train(True)
    predictor.model.sam_prompt_encoder.train(True)
    return float(np.mean(dices))

print('fine-tuning helpers ready')

In [ ]:
# -- Part B: training loop with early stopping ------------------------------
best_val_dice = -1.0
epochs_without_improve = 0
train_samples = list(train_box_ds.all_samples)

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train_samples)
    losses = []
    img_cache = {}
    for batch_start in range(0, len(train_samples), BATCH_SIZE):
        batch = train_samples[batch_start:batch_start + BATCH_SIZE]
        batch_losses = []
        for img_name, shape_idx in batch:
            base = os.path.splitext(img_name)[0]
            if img_name not in img_cache:
                img_gray = cv2.imread(os.path.join(TRAIN_IMG, img_name), cv2.IMREAD_GRAYSCALE)
                with open(os.path.join(TRAIN_JSON, base + '.json'), encoding='utf-8') as f:
                    img_cache[img_name] = (img_gray, _json.load(f))
            img_gray, data = img_cache[img_name]
            H, W = img_gray.shape
            points = np.array(data['shapes'][shape_idx]['points'])
            gt = np.zeros((H, W), dtype=np.float32)
            cv2.fillPoly(gt, [points.astype(np.int32)], 1.0)
            x_min, y_min = points.min(axis=0); x_max, y_max = points.max(axis=0)
            if random.random() < MIXED_SHIFT_PROB:
                b = train_box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W)
            else:
                b = train_box_ds._center_zoom_bbox(x_min, x_max, y_min, y_max, H, W)
            box_xyxy = [b[0], b[2], b[1], b[3]]
            batch_losses.append(sam2_compute_loss(img_gray, box_xyxy, gt))

        predictor.model.zero_grad()
        batch_loss = torch.stack(batch_losses).mean()
        scaler.scale(batch_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(batch_loss.item()))

    val_dice = validate()
    print(f'epoch {epoch:3d}/{EPOCHS} | train loss {np.mean(losses):.4f} | val Dice (off-center) {val_dice:.4f}')
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_without_improve = 0
        torch.save(predictor.model.state_dict(), BEST_CKPT_PATH)
        print(f'  -> new best, saved to {BEST_CKPT_PATH}')
    else:
        epochs_without_improve += 1
        if epochs_without_improve >= PATIENCE:
            print(f'early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)')
            break

print(f'\nBest validation Dice (off-center): {best_val_dice:.4f}')

## Part B - re-test the best fine-tuned checkpoint on the test split

Loads the best-validation-Dice checkpoint and reruns the exact same test
procedure as Part A, so the two rows are directly comparable.

In [ ]:
# -- Part B: load best checkpoint, test --
predictor.model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=True))
rows_finetuned, images_finetuned = run_test(test_box_ds, TEST_IMG, TEST_JSON, tag='fine-tuned')

# -- Qualitative figures (10 shared stems, same style as Finetune_SAMMed2D_test_robust.ipynb) --
visualize_qualitative(images_finetuned, TEST_IMG, test_box_ds, 'SAM2 (fine-tuned)',
                      f'sam2_finetuned_{DS_NAME.lower()}')

print('\n\n=== Zero-shot vs fine-tuned, side by side ===')
for rz, rf in zip(rows_zeroshot, rows_finetuned):
    assert rz['prompt'] == rf['prompt']
    print(f"{rz['prompt']:<12} zero-shot Dice {rz['dice']:.3f}  ->  fine-tuned Dice {rf['dice']:.3f}"
          f"  (delta {rf['dice']-rz['dice']:+.3f})")